# 🧹 Data Preprocessing and Cleaning

This notebook cleans and prepares the data created in the previous exploration step.

## What This Notebook Does:
1. Loads and validates the merged datasets
2. Checks for missing values and data quality issues
3. Removes unnecessary columns
4. Groups transactions by date to eliminate duplicates
5. Merges multiple datasets into a final clean file

## Output:
- `final.csv` - Clean, merged dataset ready for model training

---

## Step 1: Load Data

First, let's load the product demand dataset created in the previous notebook.

## 1. Load the Merged Data

In [11]:
import pandas as pd
import numpy as np

# Load the merged dataset from data/raw directory
df = pd.read_csv('../data/raw/products_for_ts.csv')
print("Data loaded successfully!")

Data loaded successfully!


## Step 2: Basic Dataset Information

In [12]:
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("\nColumn names:")
print(list(df.columns))

Dataset shape: (151224, 8)
Number of rows: 151224
Number of columns: 8

Column names:
['id_produit', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'date', 'quantite_demande']


In [13]:
print("First 5 rows:")
df.head()

First 5 rows:


,id_produit,categorie,colisage fardeau,colisage palette,volume pcs (m3),Is_Gerbable,date,quantite_demande
0,31334,MOULURE,32,1600,0.0002,True,2024-04-21 08:31:24,32
1,31334,MOULURE,32,1600,0.0002,True,2024-05-05 14:45:27,32
2,31334,MOULURE,32,1600,0.0002,True,2024-05-07 07:51:42,32
3,31334,MOULURE,32,1600,0.0002,True,2024-05-19 07:28:37,32
4,31334,MOULURE,32,1600,0.0002,True,2024-05-19 13:20:08,32


## Step 3: Check for Missing Values

In [14]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percentage.values
})

print("Missing Values Summary:")
print(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print("\nNo missing values found!")

Missing Values Summary:
Empty DataFrame
Columns: [Column, Missing Count, Missing Percentage]
Index: []

No missing values found!


## Step 4: Unique Values per Column

In [15]:
print("Unique values per column:\n")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} unique values")
    
    # Show unique values if there are fewer than 20
    if unique_count < 20:
        print(f"  → Values: {sorted(df[col].unique())}")
    print()

Unique values per column:

id_produit: 1124 unique values

categorie: 29 unique values

colisage fardeau: 47 unique values

colisage palette: 21 unique values

volume pcs (m3): 50 unique values

Is_Gerbable: 2 unique values
  → Values: [np.False_, np.True_]

date: 12956 unique values

quantite_demande: 778 unique values



## Step 5: Count Total Products

In [16]:
# Count unique products
if 'id_produit' in df.columns:
    n_products = df['id_produit'].nunique()
    print(f"Total number of unique products: {n_products}")
    print(f"Total number of records: {len(df)}")
    print(f"Average records per product: {len(df) / n_products:.2f}")
else:
    print("Column 'id_produit' not found in the dataset")

Total number of unique products: 1124
Total number of records: 151224
Average records per product: 134.54


## Step 6: Data Types

In [17]:
print("Data types of each column:\n")
print(df.dtypes)
print("\n" + "="*50)
print("\nData type distribution:")
print(df.dtypes.value_counts())

Data types of each column:

id_produit            int64
categorie            object
colisage fardeau      int64
colisage palette      int64
volume pcs (m3)     float64
Is_Gerbable            bool
date                 object
quantite_demande      int64
dtype: object


Data type distribution:
int64      4
object     2
float64    1
bool       1
Name: count, dtype: int64


## Step 7: Summary Statistics

In [18]:
print("Summary statistics for numerical columns:\n")
df.describe()

Summary statistics for numerical columns:



,id_produit,colisage fardeau,colisage palette,volume pcs (m3),quantite_demande
count,151224.000000,151224.000000,151224.000000,151224.000000,151224.000000
mean,32869.074386,74.724825,2047.536105,0.034886,327.534968
std,2208.848287,46.849406,627.319773,0.011811,773.053766
min,31334.000000,1.000000,1000.000000,0.000000,0.000000
25%,31587.000000,40.000000,1500.000000,0.034000,60.000000
50%,31883.000000,90.000000,2100.000000,0.037000,120.000000
75%,34015.000000,100.000000,2600.000000,0.041000,300.000000
max,45484.000000,480.000000,3000.000000,0.090000,48000.000000


## Step 8: Remove Unnecessary Columns

In [19]:
# Remove unnecessary columns
columns_to_remove = ['sku', 'nom_produit', 'unite_mesure', 'actif', 'Poids(kg)']

# Check which columns exist before dropping
existing_columns = [col for col in columns_to_remove if col in df.columns]
missing_columns = [col for col in columns_to_remove if col not in df.columns]

if existing_columns:
    df = df.drop(columns=existing_columns)
    print(f"Removed columns: {existing_columns}")
else:
    print("No columns to remove")

if missing_columns:
    print(f"Warning: These columns were not found: {missing_columns}")

print(f"\nNew dataset shape: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")

# Save the cleaned data back to CSV in data/raw directory
df.to_csv('../data/raw/products_for_ts.csv', index=False)
print("\nCleaned data saved to ../data/raw/products_for_ts.csv")

No columns to remove

New dataset shape: (151224, 8)
Remaining columns: ['id_produit', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'date', 'quantite_demande']



Cleaned data saved to ../data/raw/products_for_ts.csv


## Step 9: Group Transactions by Date

Load the transaction data and group by date to avoid duplicate entries for the same product on the same day.

In [20]:
# Reload fresh data to group properly from data/raw directory
df = pd.read_csv('../data/raw/tempo_ts_grouped.csv')

print("Original data shape:", df.shape)
print("Columns:", list(df.columns))
print("\nFirst few rows:")
print(df.head())

# Rename columns to match expected names
# 'cree_le' -> 'date', 'quantite' -> 'quantite_demande'
df = df.rename(columns={
    'cree_le': 'date',
    'quantite': 'quantite_demande'
})

print("\nColumns after renaming:", list(df.columns))

# Convert date column to datetime (auto-detect format)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Show sample dates
print("\nSample dates before grouping:")
print(df[['id_produit', 'date', 'quantite_demande']].head(10))

# Extract just the date (no time)
df['date_only'] = df['date'].dt.date

# Group by product ID and date, summing quantities
print("\n" + "="*70)
print("GROUPING BY PRODUCT AND DATE...")
print("="*70)

# Define aggregation: sum quantite_demande, keep first for other columns
agg_dict = {
    'categorie': 'first',
    'colisage fardeau': 'first',
    'colisage palette': 'first',
    'volume pcs (m3)': 'first',
    'Is_Gerbable': 'first',
    'quantite_demande': 'sum'  # SUM the quantities for same day
}

# Group and aggregate
df_grouped = df.groupby(['id_produit', 'date_only'], as_index=False).agg(agg_dict)

# Rename date_only back to date
df_grouped.rename(columns={'date_only': 'date'}, inplace=True)

print(f"\nShape after grouping: {df_grouped.shape}")
print(f"Reduced from {df.shape[0]} rows to {df_grouped.shape[0]} rows")
print(f"Removed {df.shape[0] - df_grouped.shape[0]} duplicate day entries")

print("\nFirst 10 rows of grouped data:")
print(df_grouped.head(10))

# Check if there are any duplicates by showing an example product
if df_grouped.shape[0] > 0:
    sample_product = df_grouped['id_produit'].iloc[0]
    sample_data = df_grouped[df_grouped['id_produit'] == sample_product].head(5)
    print(f"\n✓ Sample data for product {sample_product}:")
    print(sample_data[['id_produit', 'date', 'quantite_demande']])

# Save the grouped data to data/raw directory
df_grouped.to_csv('../data/raw/tempo_for_ts_final.csv', index=False)
print("\n✓ Grouped data saved to ../data/raw/tempo_for_ts_final.csv")

# Update df
df = df_grouped

Original data shape: (67825, 8)
Columns: ['id_produit', 'cree_le', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite']

First few rows:
   id_produit              cree_le        categorie  colisage fardeau  \
0       31851  2025-03-12 00:00:00           MODULE                80   
1       31501  2025-03-12 00:00:00  DISPINA METALIC               120   
2       31501  2025-03-12 00:00:00  DISPINA METALIC               120   
3       31490  2025-03-12 00:00:00  DISPINA METALIC                80   
4       31496  2025-03-12 00:00:00  DISPINA METALIC                40   

   colisage palette  volume pcs (m3)  Is_Gerbable  quantite  
0              1800            0.037         True        80  
1              2100            0.037         True       120  
2              2100            0.037         True       480  
3              2500            0.037         True       160  
4              2100            0.037         True        40  

Colum

## Step 10: Check for Duplicate Days per Product

Verify if there are still any products with multiple transactions on the same day.

In [21]:
# Load the original data to check for duplicates from data/raw directory
df_check = pd.read_csv('../data/raw/tempo_for_ts_final.csv')

# Convert date to datetime and extract date only
df_check['date'] = pd.to_datetime(df_check['date'], errors='coerce')
df_check['date_only'] = df_check['date'].dt.date

# Check for products with multiple transactions on the same day
duplicates = df_check.groupby(['id_produit', 'date_only']).size().reset_index(name='count')
duplicates_found = duplicates[duplicates['count'] > 1]

print(f"Total unique (product, date) combinations: {len(duplicates)}")
print(f"Combinations with multiple transactions: {len(duplicates_found)}")
print(f"Percentage with duplicates: {len(duplicates_found) / len(duplicates) * 100:.2f}%")

if len(duplicates_found) > 0:
    print(f"\nProducts with duplicate days:")
    print(duplicates_found.sort_values('count', ascending=False).head(10))
    
    # Show an example
    example_product = duplicates_found.iloc[0]['id_produit']
    example_date = duplicates_found.iloc[0]['date_only']
    
    print(f"\nExample: Product {example_product} on {example_date}:")
    example_rows = df_check[(df_check['id_produit'] == example_product) & (df_check['date_only'] == example_date)]
    print(example_rows[['id_produit', 'date', 'quantite_demande']])
    
    print("\n⚠️ These products need to be grouped by day to sum quantities!")
else:
    print("\n✓ All products have unique days - no grouping needed!")

Total unique (product, date) combinations: 33774
Combinations with multiple transactions: 0
Percentage with duplicates: 0.00%

✓ All products have unique days - no grouping needed!


## Step 11: Merge All Datasets

Combine `tempo_for_ts_final.csv` and `products_for_ts_grouped.csv` into a single final dataset.

In [22]:
import pandas as pd

# Load both datasets from data/raw directory
df_tempo = pd.read_csv('../data/raw/tempo_for_ts_final.csv')
df_products = pd.read_csv('../data/raw/products_for_ts_grouped.csv')

print("tempo_for_ts_final shape:", df_tempo.shape)
print("tempo_for_ts_final columns:", list(df_tempo.columns))
print("\nproducts_for_ts_grouped shape:", df_products.shape)
print("products_for_ts_grouped columns:", list(df_products.columns))

# Check if columns need renaming in df_products
# If it has 'date_demande' instead of 'date', rename it
if 'date_demande' in df_products.columns and 'date' not in df_products.columns:
    df_products = df_products.rename(columns={'date_demande': 'date'})
    print("\n✓ Renamed 'date_demande' to 'date' in products dataset")

if 'quantite' in df_products.columns and 'quantite_demande' not in df_products.columns:
    df_products = df_products.rename(columns={'quantite': 'quantite_demande'})
    print("✓ Renamed 'quantite' to 'quantite_demande' in products dataset")

# Convert date columns to date type for proper comparison
df_tempo['date'] = pd.to_datetime(df_tempo['date']).dt.date
df_products['date'] = pd.to_datetime(df_products['date']).dt.date

# Add source column to track where rows came from
df_tempo['source'] = 'tempo'
df_products['source'] = 'products'

print("\nChecking for overlaps (same product ID and date)...")

# Find overlapping rows
tempo_keys = set(zip(df_tempo['id_produit'], df_tempo['date']))
products_keys = set(zip(df_products['id_produit'], df_products['date']))

overlaps = tempo_keys.intersection(products_keys)

print(f"\nFound {len(overlaps)} overlapping (product, date) combinations")

if len(overlaps) > 0:
    print("\nFirst 10 overlapping combinations:")
    for i, (prod_id, date) in enumerate(list(overlaps)[:10]):
        print(f"  {i+1}. Product {prod_id} on {date}")
        
        # Show the different values
        tempo_row = df_tempo[(df_tempo['id_produit'] == prod_id) & (df_tempo['date'] == date)]
        products_row = df_products[(df_products['id_produit'] == prod_id) & (df_products['date'] == date)]
        
        tempo_qty = tempo_row['quantite_demande'].values[0] if not tempo_row.empty else None
        products_qty = products_row['quantite_demande'].values[0] if not products_row.empty else None
        
        if tempo_qty != products_qty:
            print(f"     ⚠️ Different quantities: tempo={tempo_qty}, products={products_qty}")
            print(f"     → Keeping tempo value: {tempo_qty}")

# Concatenate both dataframes
df_combined = pd.concat([df_tempo, df_products], ignore_index=True)

print(f"\nCombined shape before deduplication: {df_combined.shape}")

# Remove duplicates: keep first occurrence (tempo comes first, so tempo rows are kept)
df_final = df_combined.drop_duplicates(subset=['id_produit', 'date'], keep='first')

print(f"Final shape after deduplication: {df_final.shape}")
print(f"Removed {df_combined.shape[0] - df_final.shape[0]} duplicate rows")

# Drop the source column
df_final = df_final.drop(columns=['source'])

print("\nFinal columns:", list(df_final.columns))
print("\nFirst 10 rows:")
print(df_final.head(10))

# Save to final.csv in data/raw directory
df_final.to_csv('../data/raw/final.csv', index=False)
print("\n✓ Merged data saved to ../data/raw/final.csv")

# Summary
print("\n" + "="*70)
print("MERGE SUMMARY")
print("="*70)
print(f"Rows from tempo_for_ts_final: {df_tempo.shape[0]}")
print(f"Rows from products_for_ts_grouped: {df_products.shape[0]}")
print(f"Overlapping rows (duplicates): {len(overlaps)}")
print(f"Final rows in final.csv: {df_final.shape[0]}")
print(f"Total unique products: {df_final['id_produit'].nunique()}")

tempo_for_ts_final shape: (33774, 8)
tempo_for_ts_final columns: ['id_produit', 'date', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite_demande']

products_for_ts_grouped shape: (80651, 8)
products_for_ts_grouped columns: ['id_produit', 'date', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite_demande']

Checking for overlaps (same product ID and date)...

Found 21419 overlapping (product, date) combinations

First 10 overlapping combinations:
  1. Product 31507 on 2025-05-21
     ⚠️ Different quantities: tempo=80, products=160
     → Keeping tempo value: 80
  2. Product 31870 on 2026-01-07
  3. Product 31569 on 2025-11-17
     ⚠️ Different quantities: tempo=700, products=7100
     → Keeping tempo value: 700
  4. Product 31441 on 2025-11-17
     ⚠️ Different quantities: tempo=80, products=180
     → Keeping tempo value: 80
  5. Product 31760 on 2025-07-29
     ⚠️ Different quantities: tempo=

---

## ✅ Preprocessing Complete!

**Created Files:**
- `../data/raw/tempo_for_ts_final.csv` - Cleaned and grouped transaction data
- `../data/raw/final.csv` - **Final merged dataset ready for modeling**

**Next Step:** Open `03_training.ipynb` to train forecasting models on this clean data.